# PatchTST: Патчи решают

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/21_patchtst.ipynb)

## Установка зависимостей

In [ ]:
!pip install -q neuralforecast pandas numpy matplotlib

## Подготовка данных

In [ ]:
import pandas as pd
import numpy as np

# Создаём синтетические данные
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=365, freq='D')
y = 100 + np.cumsum(np.random.randn(365)) + 20 * np.sin(np.arange(365) / 7 * 2 * np.pi)

train = pd.DataFrame({
    'unique_id': 'series_1',
    'ds': dates,
    'y': y
})
print(train.head())

## PatchTST: конфигурация и обучение

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
from neuralforecast.losses.pytorch import MAE

# Параметры
HORIZON = 16
INPUT_SIZE = 96  # длина входного окна

# Конфигурация PatchTST
model = PatchTST(
    h=HORIZON,
    input_size=INPUT_SIZE,
    loss=MAE(),
    max_steps=1000,
    
    # Параметры патчинга
    patch_len=16,             # размер патча
    stride=8,                 # шаг между патчами
    
    # Архитектура трансформера
    hidden_size=128,          # размерность модели
    n_heads=4,                # количество голов внимания
    e_layers=3,               # количество слоёв encoder
    d_ff=256,                 # размерность feed-forward
    dropout=0.2,
    
    scaler_type='standard',
    random_seed=42
)

# Обучаем
nf = NeuralForecast(
    models=[model],
    freq='D'
)
nf.fit(df=train)

# Прогнозируем
forecasts = nf.predict()
print(forecasts)

## Подбор параметров патчинга

In [ ]:
def suggest_patch_params(input_size, season_length, horizon):
    """
    Эвристика для выбора параметров патчинга.
    
    Принципы:
    - Патч должен захватывать локальный паттерн (fraction сезона)
    - Stride обычно = patch_len / 2 для overlap
    - Количество патчей должно быть разумным (не слишком мало, не слишком много)
    """
    
    # Патч как доля сезона
    # Для недельной сезонности (7 дней) хороший патч — 2-3 дня
    patch_len = max(4, season_length // 3)
    
    # Округляем до степени двойки (удобно для GPU)
    patch_len = 2 ** int(np.log2(patch_len))
    
    # Stride — половина патча для 50% overlap
    stride = patch_len // 2
    
    # Проверяем, что получается разумное число патчей
    n_patches = (input_size - patch_len) // stride + 1
    
    if n_patches < 4:
        # Слишком мало патчей — уменьшаем patch_len
        patch_len = patch_len // 2
        stride = stride // 2
        n_patches = (input_size - patch_len) // stride + 1
    
    if n_patches > 64:
        # Слишком много патчей — увеличиваем stride
        stride = patch_len  # no overlap
        n_patches = (input_size - patch_len) // stride + 1
    
    return {
        'patch_len': patch_len,
        'stride': stride,
        'n_patches': n_patches
    }

# Пример для дневных данных с недельной сезонностью
params = suggest_patch_params(
    input_size=96,
    season_length=7,
    horizon=16
)
print(f"Suggested params: {params}")

## Визуализация патчей

In [ ]:
import matplotlib.pyplot as plt

def visualize_patching(series, patch_len, stride, n_patches_to_show=5):
    """
    Визуализация того, как ряд разбивается на патчи.
    """
    fig, axes = plt.subplots(n_patches_to_show + 1, 1, 
                             figsize=(12, 2 * (n_patches_to_show + 1)))
    
    # Полный ряд
    axes[0].plot(series, color='black', linewidth=1)
    axes[0].set_title('Полный ряд')
    axes[0].set_xlim(0, len(series))
    
    # Подсвечиваем патчи разными цветами
    colors = plt.cm.tab10(np.linspace(0, 1, n_patches_to_show))
    
    for i in range(n_patches_to_show):
        start = i * stride
        end = start + patch_len
        
        if end > len(series):
            break
        
        # На полном ряде
        axes[0].axvspan(start, end, alpha=0.3, color=colors[i])
        
        # Отдельный патч
        patch = series[start:end]
        axes[i + 1].plot(patch, color=colors[i], linewidth=2)
        axes[i + 1].set_title(f'Патч {i + 1}: позиции {start}-{end}')
        axes[i + 1].set_xlim(0, patch_len)
    
    plt.tight_layout()
    plt.show()

# Пример
sample_series = train['y'].values[-96:]
visualize_patching(sample_series, patch_len=16, stride=8)

## Сравнение с N-HiTS и TSMixer

In [ ]:
from neuralforecast.models import NHITS, TSMixer

models = [
    PatchTST(
        h=HORIZON,
        input_size=INPUT_SIZE,
        patch_len=16,
        stride=8,
        hidden_size=128,
        n_heads=4,
        e_layers=3,
        loss=MAE(),
        max_steps=1000,
        scaler_type='standard',
        random_seed=42
    ),
    NHITS(
        h=HORIZON,
        input_size=INPUT_SIZE,
        n_pool_kernel_size=[1, 2, 4],
        n_freq_downsample=[1, 2, 4],
        loss=MAE(),
        max_steps=1000,
        scaler_type='standard',
        random_seed=42
    )
]

nf = NeuralForecast(models=models, freq='D')
nf.fit(df=train)
forecasts = nf.predict()
print(forecasts)

## Визуализация прогнозов

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# История
history = train.tail(50)
ax.plot(history['ds'], history['y'], label='История', color='blue')

# Прогнозы
ax.plot(forecasts['ds'], forecasts['PatchTST'], 
        label='PatchTST', linestyle='--', color='red')
ax.plot(forecasts['ds'], forecasts['NHITS'], 
        label='N-HiTS', linestyle='--', color='green')

ax.set_title('PatchTST vs N-HiTS')
ax.set_xlabel('Дата')
ax.set_ylabel('Значение')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()